In [5]:
%%bash
echo "📦 1. 检测到系统重启，正在重新为你构建数字分身锻造炉..."
pip install --upgrade pip
pip install unsloth_zoo
pip install --no-deps "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
pip install --no-deps xformers trl peft accelerate bitsandbytes
echo "✅ 环境全部重装完毕！请继续运行下一步！"

📦 1. 检测到系统重启，正在重新为你构建数字分身锻造炉...
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7wxkq81v/unsloth_25031c07a9294ddea2c6c34b424429e9
  Resolved https://github.com/unslothai/unsloth.git to commit 36ea02ea817b7a85ab982ed1f362cea88ad50e4a
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
✅ 环境全部重装完毕！请继续运行下一步！


  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7wxkq81v/unsloth_25031c07a9294ddea2c6c34b424429e9


In [7]:
# 【极其重要】：在 Kaggle 原生 Cell 里，unsloth 必须是第一行！
from unsloth import FastVisionModel, is_bfloat16_supported
import torch
import os
import shutil
import trl
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainerCallback

# =====================================================================
# 1. 战报监听器
# =====================================================================
class StepLoggerCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            step = state.global_step
            max_steps = state.max_steps
            loss = logs.get("loss", 0)
            lr = logs.get("learning_rate", 0)
            print(f"🎯 [狂飙雷达] Step: {step}/{max_steps} | 📉 损失(Loss): {loss:.4f} | 学习率: {lr:.6f}")

# =====================================================================
# 2. 核心路径配置
# =====================================================================
DATASET_PATH = "/kaggle/input/datasets/xiangjiaowei/dierbuwancheng/dataset_jh_gold (3).jsonl"
OUTPUT_MODEL_DIR = "/kaggle/working/Qwen3-VL-4B-JH-Twin-HF"
GGUF_OUTPUT_DIR = "/kaggle/working/Qwen-JH-Twin-GGUF" # 专门为 GGUF 留的神圣房间

print("📥 1. 正在加载基座模型 Qwen3-VL-4B-Instruct ...")
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "Qwen/Qwen3-VL-4B-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth", 
)

print("🧠 2. 配置 PEFT 权重 (保留完整全层，保障学习质量)...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False,      
    finetune_language_layers = True,     
    finetune_attention_modules = True,   
    r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

print(f"📊 3. 正在读取数据集: {DATASET_PATH} ...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def format_chat_template(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

print("🔄 正在高速对齐语料格式...")
dataset = dataset.map(format_chat_template, batched=True, num_proc=4) 
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
if text_tokenizer.pad_token is None: text_tokenizer.pad_token = text_tokenizer.eos_token

print("⚙️ 4. 正在注入【暴力压榨版】训练参数 (Batch=8, Seq=512)...")
if hasattr(trl, "SFTConfig"):
    from trl import SFTConfig
    training_args = SFTConfig(
        per_device_train_batch_size = 8,  
        gradient_accumulation_steps = 4, 
        warmup_steps = 15, 
        num_train_epochs = 1, learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear",
        seed = 3407, output_dir = "outputs", report_to = "none",              
        max_seq_length = 512,             
        dataset_text_field = "text", 
        packing = False,                  
        dataset_num_proc = 4, dataloader_num_workers = 2 
    )
    trainer = SFTTrainer(model = model, train_dataset = dataset, processing_class = text_tokenizer, args = training_args, callbacks=[StepLoggerCallback()])
else:
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        per_device_train_batch_size = 8,  
        gradient_accumulation_steps = 4, 
        warmup_steps = 15,
        num_train_epochs = 1, learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear",
        seed = 3407, output_dir = "outputs", report_to = "none",
        dataloader_num_workers = 2
    )
    trainer = SFTTrainer(model = model, train_dataset = dataset, dataset_text_field = "text", max_seq_length = 512, packing = False, tokenizer = text_tokenizer, args = training_args, callbacks=[StepLoggerCallback()])

print("🚀 5. 暴力微调正式启动！总步数大约 360 步左右！")
trainer.train()

print("📦 6. 训练完成！正在合并为标准 16-bit 完整权重 (剥离 ZIP 打包环节)...")
model.save_pretrained_merged(OUTPUT_MODEL_DIR, tokenizer, save_method="merged_16bit")
print("✅ 16-bit 原生权重合并保存成功！(它安稳地躺在 working 文件夹里)")

print("🧹 7. [防爆盘] 删除官方底层基座缓存，腾出极其宝贵的显存和硬盘...")
shutil.rmtree("/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-4B-Instruct", ignore_errors=True)

print("🗜️ 8. 【高光时刻】正在启动神级压缩：将模型直接压制为 Q4_K_M GGUF 格式 (预计 3-5 分钟)...")
try:
    # 自动分离视觉和文本层，并进行 Q4_K_M 极致压缩
    model.save_pretrained_gguf(GGUF_OUTPUT_DIR, tokenizer, quantization_method="q4_k_m")
    print("✅ 神级压制完成！你的 GGUF 最终武器已经准备就绪！")
except Exception as e:
    print(f"⚠️ GGUF 压制遇到异常: {e}，但请放心，未压缩的 HF 权重已安全保存在第 6 步！")

print("\n✨ 【大功告成】不管怎样，现在立刻去右上角点击 Save Version -> Quick Save 永久焊死你的战利品！")

📥 1. 正在加载基座模型 Qwen3-VL-4B-Instruct ...
==((====))==  Unsloth 2026.5.2: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

⚙️ 4. 正在注入【暴力压榨版】训练参数 (Batch=8, Seq=512)...


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/11552 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:07<00:00, 33.60s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Qwen-JH-Twin-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|ERROR]Unsloth: Error during loading or introspecting the original script: Failed to execute module convert_hf_to_gguf_original_gguf_mlmglkqk from /root/.unsloth/llama.cpp/original_gguf_mlmglkqk.py
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 894, in _load_module_from_path
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/root/.unsloth/llama.cpp/original_gguf_mlmglkqk.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 957, in _download_convert_hf_to_gguf_cached
    module = _load_module_from_path(temp_original_file

⚠️ GGUF 压制遇到异常: Unsloth: GGUF conversion failed in Kaggle environment.
This is likely due to the 20GB disk space limit.
Try saving to /tmp directory or use a smaller model.
Error: Failed during loading/introspection of original script: Failed to execute module convert_hf_to_gguf_original_gguf_mlmglkqk from /root/.unsloth/llama.cpp/original_gguf_mlmglkqk.py，但请放心，未压缩的 HF 权重已安全保存在第 6 步！

✨ 【大功告成】不管怎样，现在立刻去右上角点击 Save Version -> Quick Save 永久焊死你的战利品！
